In [2]:

from google.colab import files

uploaded = files.upload()


Saving best_churn_model.pkl to best_churn_model.pkl


In [3]:

import os

print(os.listdir('/content'))


['.config', 'best_churn_model.pkl', 'sample_data']


In [4]:

!pip -q install streamlit joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.4 MB/s eta 0:00:00


In [5]:

import joblib

model_bundle = joblib.load('/content/best_churn_model.pkl')

print("Các thành phần trong file:", model_bundle.keys())
print("Các cột đầu vào của mô hình:")
print(model_bundle['feature_columns'])
print("Ngày tham chiếu:", model_bundle['ref_date'])


Các thành phần trong file: dict_keys(['model', 'ref_date', 'feature_columns'])
Các cột đầu vào của mô hình:
['country', 'age', 'gender', 'membership_tier', 'total_orders', 'total_spend_usd', 'avg_order_value_usd', 'days_since_last_purchase', 'preferred_category', 'preferred_device', 'preferred_payment_method', 'acquisition_channel', 'reviews_given', 'avg_review_score', 'returns_made', 'wishlist_items', 'newsletter_subscribed', 'tenure_days']
Ngày tham chiếu: 2026-04-01 00:00:00


#Tạo file giao diện Streamlit

In [6]:

%%writefile /content/app.py
import streamlit as st
import pandas as pd
import joblib

# Đọc mô hình đã lưu từ Task 4
bundle = joblib.load("/content/best_churn_model.pkl")
model = bundle["model"]
ref_date = pd.to_datetime(bundle["ref_date"])
feature_columns = bundle["feature_columns"]

# Cấu hình trang
st.set_page_config(
    page_title="Customer Churn Prediction",
    page_icon="📊",
    layout="wide"
)

st.title("📊 Customer Churn Prediction")
st.write(
    "Nhập thông tin khách hàng để dự đoán "
    "khả năng khách hàng rời bỏ."
)

st.info(
    "Đây là công cụ hỗ trợ phân tích. "
    "Kết quả dự đoán không đảm bảo hành vi thực tế của khách hàng."
)

# Form nhập dữ liệu
with st.form("customer_form"):
    st.subheader("Thông tin khách hàng")

    col1, col2 = st.columns(2)

    with col1:
        country = st.text_input("Country", value="United States")
        age = st.number_input("Age", min_value=18, max_value=100, value=30)
        gender = st.selectbox(
            "Gender", ["Male", "Female", "Other"]
        )
        membership_tier = st.selectbox(
            "Membership tier", ["Basic", "Silver", "Gold", "Platinum"]
        )
        registration_date = st.date_input(
            "Registration date", value=pd.Timestamp.today().date()
        )
        total_orders = st.number_input(
            "Total orders", min_value=0, value=5
        )
        total_spend_usd = st.number_input(
            "Total spend (USD)", min_value=0.0, value=250.0
        )

    with col2:
        avg_order_value_usd = st.number_input(
            "Average order value (USD)", min_value=0.0, value=50.0
        )
        days_since_last_purchase = st.number_input(
            "Days since last purchase", min_value=0, value=30
        )
        preferred_category = st.text_input(
            "Preferred category", value="Electronics"
        )
        preferred_device = st.selectbox(
            "Preferred device", ["Mobile", "Desktop", "Tablet"]
        )
        preferred_payment_method = st.selectbox(
            "Preferred payment method",
            ["Credit Card", "Debit Card", "PayPal", "Other"]
        )
        acquisition_channel = st.selectbox(
            "Acquisition channel",
            ["Organic", "Social Media", "Email", "Paid Ads", "Referral"]
        )
        reviews_given = st.number_input(
            "Reviews given", min_value=0, value=2
        )

    col3, col4 = st.columns(2)

    with col3:
        avg_review_score = st.number_input(
            "Average review score",
            min_value=0.0, max_value=5.0, value=4.0
        )
        returns_made = st.number_input(
            "Returns made", min_value=0, value=0
        )

    with col4:
        wishlist_items = st.number_input(
            "Wishlist items", min_value=0, value=3
        )
        newsletter_subscribed = st.selectbox(
            "Newsletter subscribed", [0, 1],
            format_func=lambda x: "Yes" if x == 1 else "No"
        )

    predict_button = st.form_submit_button(
        "🔍 Predict Churn", use_container_width=True
    )

if predict_button:
    # Gom dữ liệu đầu vào
    customer = {
        "country": country,
        "age": age,
        "gender": gender,
        "membership_tier": membership_tier,
        "registration_date": pd.to_datetime(registration_date),
        "total_orders": total_orders,
        "total_spend_usd": total_spend_usd,
        "avg_order_value_usd": avg_order_value_usd,
        "days_since_last_purchase": days_since_last_purchase,
        "preferred_category": preferred_category,
        "preferred_device": preferred_device,
        "preferred_payment_method": preferred_payment_method,
        "acquisition_channel": acquisition_channel,
        "reviews_given": reviews_given,
        "avg_review_score": avg_review_score,
        "returns_made": returns_made,
        "wishlist_items": wishlist_items,
        "newsletter_subscribed": newsletter_subscribed
    }

    input_df = pd.DataFrame([customer])

    # Tạo tenure_days nếu Task 4 sử dụng đặc trưng này
    if "tenure_days" in feature_columns:
        input_df["tenure_days"] = (
            ref_date - pd.to_datetime(input_df["registration_date"])
        ).dt.days

    # Chỉ giữ các cột đúng thứ tự mà mô hình yêu cầu
    missing_columns = [
        col for col in feature_columns if col not in input_df.columns
    ]

    if missing_columns:
        st.error(
            "Thiếu các cột đầu vào: " + ", ".join(missing_columns)
        )
    else:
        input_df = input_df[feature_columns]

        prediction = model.predict(input_df)[0]

        st.subheader("Kết quả dự đoán")

        if prediction == 1:
            st.error("⚠️ Dự đoán: Khách hàng có nguy cơ Churn")
        else:
            st.success("✅ Dự đoán: Khách hàng thuộc nhóm Active")

        if hasattr(model, "predict_proba"):
            probabilities = model.predict_proba(input_df)[0]
            classes = list(model.classes_)

            if 1 in classes:
                churn_probability = probabilities[classes.index(1)]
                st.metric(
                    "Xác suất dự đoán Churn",
                    f"{churn_probability:.1%}"
                )
                st.progress(float(churn_probability))

        st.caption(
            "Kết quả phụ thuộc vào dữ liệu nhập và mô hình đã huấn luyện."
        )


Writing /content/app.py


In [7]:
import os
print(os.path.exists("/content/app.py"))

True


In [8]:
!pip -q install pyngrok

In [9]:
from pyngrok import ngrok

print("Cài đặt pyngrok thành công!")

Cài đặt pyngrok thành công!


In [10]:
from pyngrok import ngrok
from getpass import getpass

token = getpass("Nhập ngrok Authtoken: ")
ngrok.set_auth_token(token)

print("Kết nối ngrok thành công!")

Nhập ngrok Authtoken: ··········
Kết nối ngrok thành công!


In [11]:
import os

print("File app.py:", os.path.exists("/content/app.py"))
print("File mô hình:", os.path.exists("/content/best_churn_model.pkl"))

File app.py: True
File mô hình: True


In [12]:
import subprocess

process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Đã khởi chạy Streamlit!")

Đã khởi chạy Streamlit!


In [13]:
from pyngrok import ngrok

ngrok.kill()
public_url = ngrok.connect(8501).public_url

print("Link ứng dụng của bạn:", public_url)

Link ứng dụng của bạn: https://handler-herald-defiant.ngrok-free.dev
